# Stellar Scatter Plots
Stellar effective temperature, surface gravity, metallicity, radius, mass, should comprise a consistent set. Associated error estimates are 1-? uncertainties.

In [ ]:
# Python imports and settings
import numpy  as np
import pandas as pd
import seaborn as sns
import scipy.stats
import re
from pydash import py_ as _
from sklearn.preprocessing import OneHotEncoder

from src.dataset_koi import koi, koi_columns, koi_column_types
from src.utilities import onehot_encode_comments

# https://stackoverflow.com/questions/11707586/how-do-i-expand-the-output-display-to-see-more-columns-of-a-pandas-dataframe
pd.set_option('display.max_columns', None)  
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', -1)
pd.set_option('display.max_rows', 8)  # 8 is required for .describe()

sns.set(rc={'figure.figsize':(20,20)})

%load_ext autoreload
%autoreload 2

In [ ]:
# cmap = sns.color_palette("YlOrRd_r", 10)
# sns.set_palette(cmap)
# sns.palplot(cmap)

In [ ]:
dataset = (
    pd.DataFrame.join(
        koi['archive'][['koi_disposition']], 
        koi['stellar']
    )
    .drop(columns='koi_sparprov')
    .sort_values(by='koi_disposition')
)
dataset

## Correlation with Laws of Physics

- koi_steff  - Stellar Effective Temperature (Kelvin) - The photospheric temperature of the star.
- koi_slogg  - Stellar Surface Gravity (log10(cm s-2)) - The base-10 logarithm of the acceleration due to gravity at the surface of the star
- koi_smet   - Stellar Metallicity - The base-10 logarithm of the Fe to H ratio at the surface of the star, normalized by the solar Fe to H ratio
- koi_srad   - Stellar Radius (solar radii) - The photospheric radius of the star
- koi_smass  - Stellar Mass (solar mass) - The mass of the star

In [ ]:
print()
print('ALL')
display(
    dataset.corr()
        .apply(lambda x: round(x,2))
        .style.background_gradient()
)
print()
print('CONFIRMED')
display(
    dataset[ dataset['koi_disposition'] == 'CONFIRMED' ].corr()
        .apply(lambda x: round(x,2))
        .style.background_gradient()
)
print()
print(disposition + ' / ALL')
display(
    dataset[ dataset['koi_disposition'] == 'CONFIRMED' ].corr()
        .divide( dataset.corr() )
        .apply(lambda x: round(x,2))
        .style.background_gradient()
)

Observations:
- As expected by the laws of physics (more fuel = hotter and bigger)
  - Stellar Mass (koi_smass) is strongly correlated with Surface Temperature (koi_steff)
  - Stellar Mass (koi_smass) is also correlated with Metallicity (koi_smet) and Radius (koi_srad)
  - Radius (koi_srad) - and by extension Stellar Mass - is anti-correlated with log10 Surface Gravity (koi_slogg) 
  
  
- Dividing correlations going from ALL -> CONFIRMED:
  - Mass/Radius = increases in correlation
  - Metallicity/Radius = uncorrelated for ALL | correlated for CONFIRMED
  - Metallicity/Gravity + Temperature/Gravity = increase in anti-correlatedness 
  - Temperature/Radius + Temperature/Metallicity = anti-correlated for ALL | correlated for CONFIRMED

## Disposition Scatter Plots

- The database of CONFIRMED KOI's may not reflect the true distribution of exoplanets, but also suffers from the selection effects of the transit method of detection

In [ ]:
print('Stellar Mass vs Radius - increases in correlation')
facetgrid = sns.relplot(
    data=dataset,    
    kind="scatter",
    col="koi_disposition", 
    col_order=['CONFIRMED', 'CANDIDATE', 'FALSE POSITIVE'],

    x="koi_smass",     
    y="koi_srad",
    size="koi_srad", sizes=(10,240),

    hue="koi_steff", palette="YlOrRd_r",

).set_axis_labels("Stellar Mass (solar mass)", "Radius (solar radii)")

Observations:
- Keplar has had most luck finding explanets around:
    - Lower mass stars (< 2 solar masses)
    - Lower temperature stars
    - Red Dwarfs + Main Sequence Stars but not Red Giants
    
    
- Possible explanations:
    - Red Giants may have gobbled up many of their planets (less exoplanets to be found)
    - Hot, large, bright stars reduce the relative dimming of a transit (exoplanets are harder to see)

In [ ]:
print('Metallicity vs Radius - correlated for CONFIRMED - uncorrelated for ALL')
sns.relplot(
    data=dataset,    
    kind="scatter",
    col="koi_disposition", 
    col_order=['CONFIRMED', 'CANDIDATE', 'FALSE POSITIVE'],
    
    x="koi_smet",     
    y="koi_srad",
    size="koi_srad", sizes=(10,240),
    
    hue="koi_steff", palette="YlOrRd_r",
).set_axis_labels("Metallicity (log10(Fe/H))", "Radius (solar radii)")

Observations:

- Exoplanets are found more often around high-metalicity stars (more material for rocky planets)
- Large Red Giants have a minimum metalicity threshold (nuclear astrophysics)    

In [ ]:
print('Temperature vs Radius - anti-correlated for ALL | correlated for CONFIRMED')
sns.relplot(
    data=dataset,    
    kind="scatter",
    col="koi_disposition", 
    col_order=['CONFIRMED', 'CANDIDATE', 'FALSE POSITIVE'],
    
    x="koi_steff",     
    y="koi_srad",
    size="koi_srad", sizes=(10,240),
    
    hue="koi_steff", palette="YlOrRd_r",
).set_axis_labels("Temperature (Kelvin)", "Radius (solar radii)")

Observations:

- Exoplanets are found more often around small low-temperature stars